In [3]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/iccad-hotspot-detection
!git clone https://github.com/Jyoti-a1230/iccad-hotspot-detection.git /content/iccad-hotspot-detection

!mkdir -p /content/raw
!unzip -q /content/drive/MyDrive/iccad_raw/iccad.zip -d /content/raw

import sys
sys.path.append('/content/iccad-hotspot-detection/src')
from data import ROOT, BENCHMARKS, label_from_filename, load_clip
from features import count_transitions

Mounted at /content/drive
Cloning into '/content/iccad-hotspot-detection'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 1), reused 19 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 120.67 KiB | 13.41 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [4]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,MaxPooling2D,Conv2D,Flatten,BatchNormalization,Dropout,ReLU,GlobalAveragePooling2D


In [5]:
model=Sequential()
model.add(Conv2D(32,kernel_size=(3,3),input_shape=(256,256,1)))
model.add(BatchNormalization())
model.add(ReLU())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64,kernel_size=(3,3)))
model.add(BatchNormalization())
model.add(ReLU())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128,kernel_size=(3,3)))
model.add(BatchNormalization())
model.add(ReLU())
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(GlobalAveragePooling2D())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 254, 254, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 125, 125, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 60, 60, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,401 (462.50 KB)

 Trainable params: 117,953 (460.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [14]:
b="iccad1"
split="train"
def load_and_label(fname, label):
    fname = fname.numpy().decode('utf-8')
    label = label.numpy().decode('utf-8')

    img = load_clip(ROOT, b, split, fname)
    img = img.resize((224, 224))
    img_arr = np.array(img)
    img_arr = np.expand_dims(img_arr, axis=-1)
    img_arr = img_arr / 255
    if label == "HS":
        labelN = 1
    else:
        labelN = 0
    return img_arr, labelN

In [8]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
import pandas as pd
import numpy as np

In [10]:
folds_df = pd.read_csv("/content/iccad-hotspot-detection/outputs/tables/train_folds.csv")
iccad1_df = folds_df[folds_df["benchmark"] == "iccad1"]

train_df = iccad1_df[iccad1_df["fold"] != 0]
val_df = iccad1_df[iccad1_df["fold"] == 0]

print(len(train_df), len(val_df))

351 88


In [11]:
def tf_wrapper(fname, label):
    img, lbl = tf.py_function(
        func=load_and_label,
        inp=[fname, label],
        Tout=[tf.float32, tf.int32]
    )
    img.set_shape((224, 224, 1))
    lbl.set_shape(())
    return img, lbl

In [12]:
train_ds = tf.data.Dataset.from_tensor_slices((train_df["filename"].values, train_df["label"].values))
train_ds = train_ds.map(tf_wrapper).batch(32).prefetch(tf.data.AUTOTUNE)

In [15]:
for images, labels in train_ds.take(1):
    print(images.shape, images.dtype)
    print(labels.shape, labels.dtype, labels.numpy())
    print(images.numpy().min(), images.numpy().max())

(32, 224, 224, 1) <dtype: 'float32'>
(32,) <dtype: 'int32'> [0 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
0.0 1.0


In [16]:
val_ds = tf.data.Dataset.from_tensor_slices((val_df["filename"].values, val_df["label"].values))
val_ds = val_ds.map(tf_wrapper).batch(32).prefetch(tf.data.AUTOTUNE)

In [17]:
for images, labels in val_ds.take(1):
    print(images.shape, images.dtype)
    print(labels.shape, labels.dtype, labels.numpy())
    print(images.numpy().min(), images.numpy().max())

(32, 224, 224, 1) <dtype: 'float32'>
(32,) <dtype: 'int32'> [1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0]
0.0 1.0


In [18]:
train_ds = tf.data.Dataset.from_tensor_slices((train_df["filename"].values, train_df["label"].values))
train_ds = train_ds.shuffle(buffer_size=len(train_df)).map(tf_wrapper).batch(32).prefetch(tf.data.AUTOTUNE)

In [19]:
for images, labels in train_ds.take(1):
    print(images.shape, images.dtype)
    print(labels.shape, labels.dtype, labels.numpy())
    print(images.numpy().min(), images.numpy().max())

(32, 224, 224, 1) <dtype: 'float32'>
(32,) <dtype: 'int32'> [0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0.0 1.0


In [20]:
!mkdir -p /content/drive/MyDrive/iccad_checkpoints

In [21]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='/content/drive/MyDrive/iccad_checkpoints/iccad1_baseline_{epoch:02d}.keras',
        save_best_only=True, monitor='val_loss'
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 29s 2s/step - loss: 0.4419 - pr_auc: 0.5167 - precision: 0.5610 - recall: 0.2911 - val_loss: 0.6325 - val_pr_auc: 0.4554 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 558ms/step - loss: 0.1725 - pr_auc: 0.9681 - precision: 0.9706 - recall: 0.8354 - val_loss: 0.5661 - val_pr_auc: 0.7490 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 602ms/step - loss: 0.0516 - pr_auc: 0.9989 - precision: 0.9870 - recall: 0.9620 - val_loss: 0.5312 - val_pr_auc: 0.7077 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 529ms/step - loss: 0.0255 - pr_auc: 0.9995 - precision: 0.9872 - recall: 0.9747 - val_loss: 0.5441 - val_pr_auc: 0.9762 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 644ms/step - loss: 0.0252 - pr_auc: 0.9984 - precision: 1.0000 - recall: 0.9873 - val_loss: 0.5901 - val_

In [22]:
from getpass import getpass
token = getpass('Paste your GitHub token: ')
username = "Jyoti-a1230"


!git config --global user.email "youractualemail@example.com"
!git config --global user.name "Jyoti-a1230"

Paste your GitHub token: ··········


In [31]:
%cd /content/iccad-hotspot-detection
!pwd

/content/iccad-hotspot-detection
/content/iccad-hotspot-detection


In [ ]:
import os

In [34]:
dirs = ["src", "notebooks", "docs", "tests", "outputs/figures", "outputs/tables", "outputs/models"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

In [35]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
import shutil
import os

# Define your source and destination paths
source_path = "/content/drive/MyDrive/Colab Notebooks/Day4.ipynb"
destination_dir = "/content/iccad-hotspot-detection/notebooks"
destination_path = os.path.join(destination_dir, "Day4.ipynb")

# Verify the source file exists before copying
if os.path.exists(source_path):
    # Ensure the destination folder exists
    os.makedirs(destination_dir, exist_ok=True)

    # Copy the notebook
    shutil.copy(source_path, destination_path)
    print(f" Successfully copied 'Day4.ipynb' to: {destination_path}")
else:
    print("❌ Could not find the file. Please verify your Google Drive is mounted and the filename is exact.")


In [36]:
!git add .
!git commit